In [0]:
from pyspark.sql.functions import col, to_date

fact_df = spark.table("medical_project.gold.fact_encounters")

display(fact_df)

In [0]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

window_spec = Window.partitionBy("patient_id").orderBy("start")

df = fact_df.withColumn(
    "prev_end",
    lag("stop").over(window_spec)
)

In [0]:
from pyspark.sql.functions import datediff

df = df.withColumn(
    "days_between",
    datediff(col("start"), col("prev_end"))
)

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "is_eligible",
    when(
        (col("prev_end").isNotNull()) &
        (col("start") >= col("prev_end")),
        1
    ).otherwise(0)
)

df = df.withColumn(
    "is_readmission",
    when(
        (col("is_eligible") == 1) &
        (col("days_between") <= 30),
        1
    ).otherwise(0)
)

display(df)

In [0]:
date_df = spark.table("medical_project.gold.dim_date")

df = df.withColumn(
    "encounter_date",
    to_date(col("start"))
).join(
    date_df,
    col("encounter_date") == col("date"),
    "left"
)

In [0]:
from pyspark.sql.functions import sum

agg_df = df.groupBy("year", "month").agg(
    sum("is_eligible").alias("eligible_encounters"),
    sum("is_readmission").alias("readmissions")
)

agg_df = agg_df.withColumn(
    "readmission_rate",
    (col("readmissions") / col("eligible_encounters")) * 100
)

In [0]:
kpi6 = agg_df.orderBy("year", "month")

display(kpi6)

In [0]:
kpi6.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.kpi_readmission_rate")